# P1 Initial Reconnaissance

## tl;dr

- Train has **776,706** rows and test has **169,011** rows; the four-column key is complete and unique.
- Train contains **32,126 positives (4.1362%)** distributed across contiguous 10-minute anomaly runs.
- Test/sample/baseline keys match exactly and in the same order.
- The organizer baseline predicts **4,402 positives (2.6046%)**; 699 of its 1,199 positive runs are singletons.
- Random row splits are unsafe. Validation must preserve time, station/layer groups, observation gaps, and anomaly-run boundaries.
- This notebook reads only the supplied local files, displays aggregate statistics only, and does not train or submit a model.

## Context & Methods

This reader-facing notebook verifies the P1 data contract, missingness, time cadence, label/run structure, and baseline shape. It is a read-only reconnaissance artifact for model and validation design.

### Key Assumptions

- The immutable directory selected by P1_DATA_DIR, or the unique project search fallback, is the authoritative input.
- station, year, layer, time is the row key.
- Times are KST (+09:00) and a 10-minute cadence is expected only within an active operating segment.
- Real sensor gaps are allowed and must break rolling features and interval post-processing.
- External KORS/KHOA values are not loaded. The current policy is no external data until written organizer approval.

Source rules: local dataset README.md, 00_MUST_READ_FIRST.md, and the KIMST notice linked in the project README.

In [1]:
from pathlib import Path
import hashlib
import os

import numpy as np
import pandas as pd
from IPython.display import display

KEY = ["station", "year", "layer", "time"]
GROUP = ["station", "year", "layer"]
REQUIRED_INPUTS = {
    "train.csv",
    "test.csv",
    "sample_submission.csv",
    "baseline_rule.csv",
    "README.md",
    "score.py",
}

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "00_MUST_READ_FIRST.md").is_file():
            return candidate
    raise FileNotFoundError("Project root marker 00_MUST_READ_FIRST.md was not found.")

def validate_input_dir(candidate: Path) -> list[str]:
    return sorted(name for name in REQUIRED_INPUTS if not (candidate / name).is_file())

def resolve_data_dir(project_root: Path) -> tuple[Path, str]:
    configured = os.environ.get("P1_DATA_DIR")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        missing = validate_input_dir(candidate)
        if missing:
            raise FileNotFoundError(f"P1_DATA_DIR is missing required files: {missing}")
        return candidate, "P1_DATA_DIR"

    matches = sorted({
        path.parent.resolve()
        for path in project_root.rglob("train.csv")
        if not validate_input_dir(path.parent)
    })
    if len(matches) != 1:
        raise RuntimeError(
            "Project search must find exactly one P1 input directory; "
            f"found {len(matches)}. Set P1_DATA_DIR explicitly."
        )
    return matches[0], "project rglob fallback"

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR, DATA_SOURCE_MODE = resolve_data_dir(PROJECT_ROOT)

pd.Series({
    "source_mode": DATA_SOURCE_MODE,
    "directory_name": DATA_DIR.name,
    "required_files_present": True,
    "raw_rows_displayed": False,
}, name="input_contract")

source_mode                 P1_DATA_DIR
directory_name            P1_qc_anomaly
required_files_present             True
raw_rows_displayed                False
Name: input_contract, dtype: object

## Data

The next cells fingerprint and load the four distributed CSV files. Outputs are bounded to file metadata and aggregate summaries; no observation rows are displayed.

In [2]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

csv_names = ["train.csv", "test.csv", "sample_submission.csv", "baseline_rule.csv"]
manifest = pd.DataFrame({
    "bytes": [(DATA_DIR / name).stat().st_size for name in csv_names],
    "sha256": [sha256_file(DATA_DIR / name) for name in csv_names],
}, index=csv_names)
display(manifest)

train = pd.read_csv(DATA_DIR / "train.csv", low_memory=False)
test = pd.read_csv(DATA_DIR / "test.csv", low_memory=False)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", low_memory=False)
baseline = pd.read_csv(DATA_DIR / "baseline_rule.csv", low_memory=False)
frames = {"train": train, "test": test, "sample": sample, "baseline": baseline}

inventory = pd.DataFrame({
    name: {
        "rows": len(frame),
        "columns": len(frame.columns),
        "period_start": frame["time"].min(),
        "period_end": frame["time"].max(),
    }
    for name, frame in frames.items()
}).T
inventory

,bytes,sha256
train.csv,50584654,20b656b0cbd524ad9da0bae8ecb6e0bacfc006e05810b3...
test.csv,10353810,6d5c6522c282651b99f4261ffa803cf99950596028e996...
sample_submission.csv,7267517,e7027bcb56836587715e5cd818c6d595ebef4a8518538f...
baseline_rule.csv,7267517,0d0de2c89fdcec8330ad09b54c80db71342da8b835aa42...


,rows,columns,period_start,period_end
train,776706,9,2024-01-01T09:00:00+09:00,2025-12-10T03:00:00+09:00
test,169011,7,2026-01-01T00:00:00+09:00,2026-06-30T23:50:00+09:00
sample,169011,6,2026-01-01T00:00:00+09:00,2026-06-30T23:50:00+09:00
baseline,169011,6,2026-01-01T00:00:00+09:00,2026-06-30T23:50:00+09:00


## Results

### 1. Schema, keys, and submission alignment

In [3]:
expected_columns = {
    "train": ["station", "year", "layer", "time", "temp", "psal", "depth", "label", "anomaly_type"],
    "test": ["station", "year", "layer", "time", "temp", "psal", "depth"],
    "sample": ["station", "year", "layer", "time", "label", "anomaly_type"],
    "baseline": ["station", "year", "layer", "time", "label", "anomaly_type"],
}
expected_rows = {"train": 776_706, "test": 169_011, "sample": 169_011, "baseline": 169_011}

schema_profile = []
for name, frame in frames.items():
    record = {
        "dataset": name,
        "rows_match_readme": len(frame) == expected_rows[name],
        "schema_exact": list(frame.columns) == expected_columns[name],
        "key_null_rows": int(frame[KEY].isna().any(axis=1).sum()),
        "duplicate_keys": int(frame.duplicated(KEY).sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
    }
    schema_profile.append(record)
    assert record["rows_match_readme"]
    assert record["schema_exact"]
    assert record["key_null_rows"] == 0
    assert record["duplicate_keys"] == 0

assert set(train["label"].dropna().unique()) <= {0, 1}
assert set(baseline["label"].dropna().unique()) <= {0, 1}
assert sample[KEY].equals(test[KEY])
assert baseline[KEY].equals(test[KEY])

display(pd.DataFrame(schema_profile).set_index("dataset"))
pd.Series({
    "sample_keys_equal_test_in_order": sample[KEY].equals(test[KEY]),
    "baseline_keys_equal_test_in_order": baseline[KEY].equals(test[KEY]),
    "train_labels_are_binary": set(train["label"].dropna().unique()) <= {0, 1},
    "baseline_labels_are_binary": set(baseline["label"].dropna().unique()) <= {0, 1},
}, name="submission_alignment")

,rows_match_readme,schema_exact,key_null_rows,duplicate_keys,exact_duplicate_rows
dataset,,,,,
train,True,True,0,0,0
test,True,True,0,0,0
sample,True,True,0,0,0
baseline,True,True,0,0,0


sample_keys_equal_test_in_order      True
baseline_keys_equal_test_in_order    True
train_labels_are_binary              True
baseline_labels_are_binary           True
Name: submission_alignment, dtype: bool

### 2. Missingness and structural exceptions

In [4]:
missing_counts = pd.concat(
    {name: frame.isna().sum() for name, frame in {"train": train, "test": test}.items()},
    axis=1,
)
missing_rates = missing_counts.div(pd.Series({"train": len(train), "test": len(test)}), axis=1)
missing_summary = pd.concat({"count": missing_counts, "rate": missing_rates}, axis=1)
display(missing_summary.loc[["temp", "psal", "depth", "label", "anomaly_type"]].fillna(0))

test_missing_by_station = test.groupby("station").agg(
    rows=("time", "size"),
    psal_missing=("psal", lambda values: int(values.isna().sum())),
    depth_missing=("depth", lambda values: int(values.isna().sum())),
)
test_missing_by_station["psal_missing_rate"] = test_missing_by_station["psal_missing"] / test_missing_by_station["rows"]
test_missing_by_station["depth_missing_rate"] = test_missing_by_station["depth_missing"] / test_missing_by_station["rows"]
test_missing_by_station

count               rate          
               train     test     train      test
temp               0      0.0  0.000000  0.000000
psal           16725    798.0  0.021533  0.004722
depth           1130  16368.0  0.001455  0.096846
label              0      0.0  0.000000  0.000000
anomaly_type  744580      0.0  0.958638  0.000000

,rows,psal_missing,depth_missing,psal_missing_rate,depth_missing_rate
station,,,,,
G-ORS,16331,245,16331,0.015002,1.000000
I-ORS,74953,516,1,0.006884,0.000013
S-ORS,77727,37,36,0.000476,0.000463


### 3. Time cadence and official operating exceptions

In [5]:
def with_parsed_time(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.assign(_time=pd.to_datetime(frame["time"], errors="coerce", utc=True))

def cadence_summary(frame: pd.DataFrame) -> pd.Series:
    parsed = with_parsed_time(frame).sort_values(GROUP + ["_time"], kind="stable")
    delta_minutes = parsed.groupby(GROUP, sort=False)["_time"].diff().dt.total_seconds().div(60).dropna()
    return pd.Series({
        "timestamp_parse_failures": int(parsed["_time"].isna().sum()),
        "exact_10_min_rate": float(delta_minutes.eq(10).mean()),
        "gaps_over_10_min": int(delta_minutes.gt(10).sum()),
        "nonpositive_deltas": int(delta_minutes.le(0).sum()),
        "operating_groups": int(parsed.groupby(GROUP, sort=False).ngroups),
    })

display(pd.concat({"train": cadence_summary(train), "test": cadence_summary(test)}, axis=1))

p2_protected_rows = train.loc[
    train["station"].eq("S-ORS")
    & train["year"].eq(2025)
    & train["layer"].isin([2, 3, 4])
    & train["time"].between("2025-09-01T00:00:00+09:00", "2025-10-31T23:59:59+09:00")
]

exception_checks = pd.Series({
    "all_time_suffixes_are_+09:00": bool(train["time"].str.endswith("+09:00").all() and test["time"].str.endswith("+09:00").all()),
    "gors_2026_rows": int(test["station"].eq("G-ORS").sum()),
    "gors_2026_depth_missing_rows": int(test.loc[test["station"].eq("G-ORS"), "depth"].isna().sum()),
    "gors_2026_depth_all_missing": bool(test.loc[test["station"].eq("G-ORS"), "depth"].isna().all()),
    "iors_2026_layer3_rows": int((test["station"].eq("I-ORS") & test["layer"].eq(3)).sum()),
    "p2_protected_sors_rows_in_train": int(len(p2_protected_rows)),
})
exception_checks

,train,test
timestamp_parse_failures,0.000000,0.000000
exact_10_min_rate,0.998443,0.994592
gaps_over_10_min,1209.000000,914.000000
nonpositive_deltas,0.000000,0.000000
operating_groups,23.000000,15.000000


all_time_suffixes_are_+09:00        True
gors_2026_rows                     16331
gors_2026_depth_missing_rows       16331
gors_2026_depth_all_missing         True
iors_2026_layer3_rows                  0
p2_protected_sors_rows_in_train        0
dtype: object

### 4. Label balance and anomaly-run structure

In [6]:
def ordered_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return with_parsed_time(frame).sort_values(GROUP + ["_time"], kind="stable")

def contiguous_run_lengths(ordered: pd.DataFrame, active: pd.Series) -> pd.Series:
    active = active.astype(bool)
    group_change = ordered[GROUP].ne(ordered[GROUP].shift()).any(axis=1)
    delta_minutes = ordered.groupby(GROUP, sort=False)["_time"].diff().dt.total_seconds().div(60)
    starts = active & (~active.shift(fill_value=False) | group_change | delta_minutes.ne(10))
    run_id = starts.cumsum()
    return active.loc[active].groupby(run_id.loc[active]).size().astype(int)

ordered_train = ordered_frame(train)
train_positive_runs = contiguous_run_lengths(ordered_train, ordered_train["label"].eq(1))

label_summary = pd.Series({
    "normal_rows": int(train["label"].eq(0).sum()),
    "positive_rows": int(train["label"].eq(1).sum()),
    "positive_rate": float(train["label"].mean()),
    "positive_runs": int(len(train_positive_runs)),
    "positive_missing_anomaly_type": int((train["label"].eq(1) & train["anomaly_type"].isna()).sum()),
    "normal_with_anomaly_type": int((train["label"].eq(0) & train["anomaly_type"].notna()).sum()),
})
display(label_summary)

base_types = ["spike", "noise", "flatline", "offset", "drift"]
type_tokens = train["anomaly_type"].fillna("").str.get_dummies(sep="+").reindex(columns=base_types, fill_value=0)
ordered_tokens = type_tokens.reindex(ordered_train.index)
type_records = []
for anomaly_type in base_types:
    run_lengths = contiguous_run_lengths(ordered_train, ordered_tokens[anomaly_type].eq(1))
    type_records.append({
        "anomaly_type": anomaly_type,
        "membership_rows": int(type_tokens[anomaly_type].sum()),
        "runs": int(len(run_lengths)),
        "min_rows_per_run": int(run_lengths.min()),
        "max_rows_per_run": int(run_lengths.max()),
    })

pd.DataFrame(type_records).set_index("anomaly_type")

normal_rows                      744580.000000
positive_rows                     32126.000000
positive_rate                         0.041362
positive_runs                       263.000000
positive_missing_anomaly_type         0.000000
normal_with_anomaly_type              0.000000
dtype: float64

,membership_rows,runs,min_rows_per_run,max_rows_per_run
anomaly_type,,,,
spike,104,104,1,1
noise,9656,52,23,353
flatline,6441,55,12,283
offset,7507,33,48,519
drift,8929,30,101,519


### 5. Organizer baseline sequence profile

In [7]:
ordered_baseline = ordered_frame(baseline)
baseline_runs = contiguous_run_lengths(ordered_baseline, ordered_baseline["label"].eq(1))

baseline_summary = pd.Series({
    "rows": len(baseline),
    "positive_rows": int(baseline["label"].sum()),
    "positive_rate": float(baseline["label"].mean()),
    "positive_runs": int(len(baseline_runs)),
    "singleton_runs": int(baseline_runs.eq(1).sum()),
    "singleton_run_rate": float(baseline_runs.eq(1).mean()),
    "longest_run_rows": int(baseline_runs.max()),
    "longest_run_hours": float(baseline_runs.max() / 6),
})
display(baseline_summary)

baseline_by_station_layer = baseline.groupby(["station", "layer"]).agg(
    rows=("label", "size"),
    positives=("label", "sum"),
    positive_rate=("label", "mean"),
)
baseline_by_station_layer

rows                  169011.000000
positive_rows           4402.000000
positive_rate              0.026046
positive_runs           1199.000000
singleton_runs           699.000000
singleton_run_rate         0.582986
longest_run_rows         232.000000
longest_run_hours         38.666667
dtype: float64

rows  positives  positive_rate
station layer                                 
G-ORS   1      16331        273       0.016717
I-ORS   1      16967        469       0.027642
        2       6342         76       0.011984
        4       6333        131       0.020685
        5      19456        636       0.032689
        6       5888        166       0.028193
        7      19967        537       0.026894
S-ORS   1      15208        490       0.032220
        2       5476        100       0.018262
        3       5782        157       0.027153
        4       6435        121       0.018803
        5      16795        683       0.040667
        6       6020        107       0.017774
        7       7343        229       0.031186
        8      14668        227       0.015476

### 6. Aggregate signal contrast

In [8]:
ordered_signal = ordered_train.copy()
grouped_signal = ordered_signal.groupby(GROUP, sort=False)
ordered_signal["abs_temp_diff"] = grouped_signal["temp"].diff().abs()
ordered_signal["same_as_previous_temp"] = grouped_signal["temp"].diff().eq(0)

signal_contrast = pd.DataFrame({
    "abs_temp_diff_mean": ordered_signal.groupby("label")["abs_temp_diff"].mean(),
    "abs_temp_diff_p99": ordered_signal.groupby("label")["abs_temp_diff"].quantile(0.99),
    "same_as_previous_temp_rate": ordered_signal.groupby("label")["same_as_previous_temp"].mean(),
})
signal_contrast.index = signal_contrast.index.map({0: "normal", 1: "anomaly"})
signal_contrast

,abs_temp_diff_mean,abs_temp_diff_p99,same_as_previous_temp_rate
label,,,
normal,0.228535,2.859844,0.002012
anomaly,1.003534,11.561425,0.194827


## Takeaways

1. **Data contract is usable:** source row counts, schemas, keys, label domains, and test submission alignment pass strict checks.
2. **Missingness is structural:** G-ORS 2026 depth is entirely missing and must not become a direct anomaly indicator.
3. **Validation must be blocked:** non-10-minute gaps and multi-hour anomaly runs make random row CV leakage-prone. Use chronological/grouped folds with purge and embargo.
4. **Type-aware features are justified:** exact repeats strongly distinguish flatlines, while robust differences and multi-scale residuals target spike, noise, offset, and drift.
5. **Post-processing is promising but type-specific:** the baseline has many singleton runs, yet true spike events are also singletons, so one global minimum-duration rule would destroy spike recall.
6. **Do not infer test prevalence:** thresholds and interval rules belong exclusively to OOF validation.
7. **External and future-context features remain off:** written organizer approval is required before either is evaluated.